<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# ML-3 : Entraînement et AutoML

**Navigation** : [Index](README.md) | [<< ML-2](ML-2-Data%26Features.ipynb) | [Suivant >>](ML-4-Evaluation.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Entraîner des modèles avec différents algorithmes (SDCA, LightGBM)
2. Utiliser AutoML pour la recherche automatique d'hyperparamètres
3. Comparer les performances de plusieurs algorithmes
4. Choisir le meilleur modèle pour un problème donné

### Prérequis
- ML-2-Data&Features complété
- Comprendre les concepts de features et pipelines

### Durée estimée : 45-60 minutes

---

# Entraînement et AutoML

## Dans ce Notebook, vous allez apprendre
- Qu'est-ce que l'**entraînement** ?
- Introduction aux entraîneurs, quelques-unes de leurs différences, et comment choisir lequel utiliser.
- Comment les hyperparamètres impactent les performances de l'entraînement.
- Comment utiliser AutoML pour simplifier votre processus d'entraînement.

## Qu'est-ce que l'**entraînement** ?
Avant de plonger dans le code, parlons d'abord de ce que signifie "entraîner un modèle".

Dans ML.Net, "entraîner un modèle" signifie généralement appeler `model.Fit(X)` dans ML.Net, où `X` est un `IDataView` qui inclut à la fois les caractéristiques et les étiquettes. Alors, que se passe-t-il lorsque vous appelez `Fit` ? En général, `Fit` met à jour les paramètres dans l'entraîneur afin qu'il puisse prédire une étiquette qui soit **proche** de l'étiquette réelle dans `X`, ou en d'autres termes, réduire la distance entre l'étiquette prédite et l'étiquette réelle.

En apprentissage automatique, la différence ou la distance entre l'étiquette prédite et l'étiquette réelle est généralement appelée **perte** et vous utilisez différentes mesures de perte en fonction de la tâche. Pour la classification, softmax est une mesure de perte courante. Pour la régression, l'erreur quadratique moyenne (RMSE) est une mesure de perte courante. En général, elles sont toutes des métriques pour quantifier la distance entre l'étiquette prédite et l'étiquette réelle. Dans la plupart des cas, une **perte plus faible signifie un meilleur modèle**. Pour plus d'informations, consultez le [guide des métriques d'évaluation ML.NET](https://docs.microsoft.com/dotnet/machine-learning/resources/metrics).

Ainsi, ce que fait `Fit` est d'appliquer un algorithme à vos données pour identifier des motifs et ajuster les paramètres dans cet algorithme pour réduire la perte. Lorsque vous entraînez un modèle, vous voulez réduire sa perte pour rendre la prédiction de ce modèle plus proche de l'étiquette réelle.

In [1]:
#i "nuget:https://pkgs.dev.azure.com/dnceng/public/_packaging/MachineLearning/nuget/v3/index.json"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Microsoft.ML.AutoML, 0.23.0"
#r "nuget: Microsoft.Data.Analysis, 0.23.0"
#r "nuget: Plotly.NET.Interactive, 3.0.2"
#r "nuget: Plotly.NET.CSharp, 0.0.1"

// Import usings.
using Microsoft.Data.Analysis;
using System;
using System.IO;
using Microsoft.ML;
using Microsoft.ML.AutoML;
using Microsoft.ML.Data;

The below script needs to be able to find the current output cell; this is an easy method to get it.

Restore sources https://pkgs.dev.azure.com/dnceng/public/_packaging/MachineLearning/nuget/v3/index.json Installed Packages Microsoft.Data.Analysis, 0.23.0 Microsoft.ML, 5.0.0 Microsoft.ML.AutoML, 0.23.0 Plotly.NET.CSharp, 0.0.1 Plotly.NET.Interactive, 3.0.2

Loading extensions from `C:\Users\jsboi\.nuget\packages\plotly.net.interactive\3.0.2\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Loading extensions from `C:\Users\jsboi\.nuget\packages\microsoft.data.analysis\0.23.0\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

Loading extensions from `C:\Users\jsboi\.nuget\packages\microsoft.ml.automl\0.23.0\interactive-extensions\dotnet\Microsoft.ML.AutoML.Interactive.dll`

Loading extensions from `C:\Users\jsboi\.nuget\packages\skiasharp\2.88.8\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

## Les entraîneurs dans ML.Net
ML.NET fournit une variété d'entraîneurs. Vous pouvez en trouver la plupart sous le [StandardTrainersCatalog](https://docs.microsoft.com/dotnet/api/microsoft.ml.standardtrainerscatalog?view=ml-dotnet). Exemples d'entraîneurs : des entraîneurs linéaires comme `SDCA`, `Lbfgs`, `LinearSvm` et des entraîneurs non linéaires basés sur les arbres comme `FastTree`, `RandomForest` et `LightGbm`. En général, les capacités de chaque entraîneur sont différentes. Les modèles non linéaires ont parfois de meilleures performances d'entraînement (perte plus faible) que les modèles linéaires, mais cela ne signifie pas toujours qu'ils sont toujours le meilleur choix. Choisir le bon entraîneur pour construire le meilleur modèle pour vos données nécessite de nombreux essais et erreurs.

### Surapprentissage et sous-apprentissage
Le surapprentissage et le sous-apprentissage sont les deux problèmes les plus courants que vous rencontrez lors de l'entraînement d'un modèle. Le sous-apprentissage signifie que l'entraîneur sélectionné n'est pas assez performant pour ajuster le jeu de données d'entraînement et résulte généralement en une perte élevée pendant l'entraînement et un score/métrique faible sur le jeu de test. Pour résoudre ce problème, vous devez soit sélectionner un modèle plus puissant, soit effectuer davantage d'ingénierie des fonctionnalités. Le surapprentissage est l'inverse, ce qui se produit lorsque le modèle apprend trop bien les données d'entraînement. Cela résulte généralement en une perte faible pendant l'entraînement mais une perte élevée sur le jeu de test.

Une bonne analogie pour ces concepts est l'étude pour un examen. Disons que vous connaissiez à l'avance les questions et les réponses. Après avoir étudié, vous passez l'examen et obtenez un score parfait. Bonne nouvelle ! Cependant, lorsqu'on vous donne à nouveau l'examen avec les questions réarrangées et légèrement reformulées, vous obtenez un score plus bas. Cela suggère que vous avez mémorisé les réponses sans vraiment apprendre les concepts sur lesquels vous étiez évalué. C'est un exemple de surapprentissage. Le sous-apprentissage est l'inverse, où les matériaux d'étude que vous avez reçus ne représentent pas précisément ce sur quoi vous êtes évalué pour l'examen. En conséquence, vous devez deviner les réponses car vous n'avez pas suffisamment de connaissances pour répondre correctement.

### Différence entre paramètre et hyper-paramètre
En résumé, les paramètres sont internes à un entraîneur et sont mis à jour en fonction du jeu de données d'entraînement pendant le processus d'entraînement (`Fit`). Les hyper-paramètres sont externes à un entraîneur et contrôlent le processus d'entraînement. Par exemple, dans `LightGbm`, le `LearningRate` est un hyper-paramètre que vous pouvez désigner lors de la création et il contrôle les étapes de mise à jour pour le poids des nœuds de l'arbre pendant l'entraînement. Et le poids des nœuds de l'arbre est un paramètre qui est ajusté pendant le processus `Fit`.

### Optimisation des hyper-paramètres
Choisir le bon entraîneur impacte vos performances finales d'entraînement. Choisir les bons hyper-paramètres a également un impact énorme sur les performances finales de l'entraînement, en particulier pour les entraîneurs basés sur les arbres. Les hyper-paramètres sont importants car ils contrôlent comment les paramètres sont mis à jour. Par exemple, un plus grand `numberOfLeaves` dans `LightGbm` produit un modèle plus grand et permet généralement de s'adapter à un jeu de données plus complexe, mais cela peut avoir un effet contraire sur un petit jeu de données et provoquer un **surapprentissage**. À l'inverse, si le jeu de données est complexe mais que vous définissez un petit `numberOfLeaves`, cela peut nuire à la capacité de `LightGbm` à s'adapter à ce jeu de données et provoquer un **sous-apprentissage**.

Le processus de recherche de la meilleure configuration pour votre entraîneur est connu sous le nom d'optimisation des hyper-paramètres (HPO). Comme le processus de choix de votre entraîneur, il implique beaucoup d'essais et d'erreurs. Les capacités intégrées de ML automatisé (AutoML) dans ML.NET simplifient le processus de HPO.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
var rand = new Random(0);
var context =new MLContext(seed: 1);
var x = Enumerable.Range(-10000, 10000).Select(_x => _x * 0.1f).ToArray();
var y = x.Select(_x => 100 * _x + (rand.NextSingle() - 0.5f) * 10).ToArray();
var df = new DataFrame();
df["X"] = DataFrameColumn.Create("X", x);
df["y"] = DataFrameColumn.Create("y", y);
var trainTestSplit = context.Data.TrainTestSplit(df);
df.Head(10)


Error: System.IO.FileNotFoundException: Could not load file or assembly 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'. Le fichier spécifié est introuvable.
File name: 'System.Collections.Immutable, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a'
   at Microsoft.ML.DataOperationsCatalog.CreateSplitColumn(IHostEnvironment env, IDataView& data, String samplingKeyColumn, Nullable`1 seed, Boolean fallbackInEnvSeed)
   at Microsoft.ML.DataOperationsCatalog.TrainTestSplit(IDataView data, Double testFraction, String samplingKeyColumnName, Nullable`1 seed)
   at Submission#5.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

## Exemple 1 : Régression linéaire
Dans la section ci-dessous, nous allons montrer la différence entre les entraîneurs via une tâche de régression linéaire. Tout d'abord, nous ajustons le jeu de données linéaire avec l'entraîneur linéaire `SDCA`. Ensuite, nous ajustons le jeu de données linéaire avec `LightGbm`, un entraîneur non linéaire basé sur les arbres. Leurs performances sont évaluées par rapport à un jeu de test. Le code ci-dessous :
- Crée un jeu de données linéaire et le divise en ensembles d'entraînement et de test
- Crée des pipelines d'entraînement utilisant `SDCA` et `LightGbm`
- Entraîne `SDCA` et `LightGbm` sur le jeu d'entraînement linéaire, et les évalue sur le jeu de test.


In [3]:
var sdcaPipeline = context.Transforms.Concatenate("Features", "X")
                            .Append(context.Regression.Trainers.Sdca("y"));

var lgbmPipeline = context.Transforms.Concatenate("Features", "X")
                            .Append(context.Regression.Trainers.LightGbm("y"));

var sdcaModel = sdcaPipeline.Fit(trainTestSplit.TrainSet);
var lgbmModel = lgbmPipeline.Fit(trainTestSplit.TrainSet);

// évaluation sur le jeu d'entraînement
var sdcaTrainEval = sdcaModel.Transform(trainTestSplit.TrainSet);
var sdcaTrainMetric = context.Regression.Evaluate(sdcaTrainEval, "y");

var lgbmTrainEval = lgbmModel.Transform(trainTestSplit.TrainSet);
var lgbmTrainMetric = context.Regression.Evaluate(lgbmTrainEval, "y");

Console.WriteLine($"sdca rmse sur trainset: {sdcaTrainMetric.RootMeanSquaredError}, lgbm rmse sur trainset: {lgbmTrainMetric.RootMeanSquaredError}");

// évaluation sur le jeu de test
var sdcaTestEval = sdcaModel.Transform(trainTestSplit.TestSet);
var sdcaTestMetric = context.Regression.Evaluate(sdcaTestEval, "y");

var lgbmTestEval = lgbmModel.Transform(trainTestSplit.TestSet);
var lgbmTestMetric = context.Regression.Evaluate(lgbmTestEval, "y");
Console.WriteLine($"sdca rmse sur testset: {sdcaTestMetric.RootMeanSquaredError}, lgbm rmse sur testset: {lgbmTestMetric.RootMeanSquaredError}");


[Cellule non executee - erreur environnement .NET]


## Exemple 2 : Régression non linéaire sur LightGbm
Cet exemple montre l'importance de l'optimisation des hyper-paramètres. Tout d'abord, nous créons un jeu de données non linéaire et deux pipelines. Un pipeline a `LightGbm` avec `numberOfLeaves` défini à `10`, l'autre est défini à `1000`. Les deux pipelines sont entraînés avec le même jeu de données d'entraînement et leurs performances sont évaluées sur le même jeu de test.

## Créer un jeu de données non linéaire
Le code ci-dessous crée un jeu de données non linéaire avec un résidu aléatoire. Le jeu de données est chargé dans un ensemble d'entraînement et un ensemble de test.


In [4]:
var rand = new Random(0);
var context = new MLContext(seed: 1);
var x = Enumerable.Range(-10000, 10000).Select(_x => _x * 0.1f).ToArray();
var y = x.Select(_x => 100 * _x * _x + (rand.NextSingle() - 0.5f) * 10).ToArray();
var df = new DataFrame();
df["X"] = DataFrameColumn.Create("X", x);
df["y"] = DataFrameColumn.Create("y", y);
var trainTestSplit = context.Data.TrainTestSplit(df);
df.Head(10);

var smallLgbmPipeline = context.Transforms.Concatenate("Features", "X")
                            .Append(context.Regression.Trainers.LightGbm("y", numberOfLeaves: 10));

var largeLgbmPipeline = context.Transforms.Concatenate("Features", "X")
                            .Append(context.Regression.Trainers.LightGbm("y", numberOfLeaves: 1000));

var smallLgbmModel = smallLgbmPipeline.Fit(trainTestSplit.TrainSet);
var largeLgbmModel = largeLgbmPipeline.Fit(trainTestSplit.TrainSet);

// évaluation sur le jeu de test
var smallTestEval = smallLgbmModel.Transform(trainTestSplit.TrainSet);
var smallLgbmMetric = context.Regression.Evaluate(smallTestEval, "y");

var largeLgbmEval = largeLgbmModel.Transform(trainTestSplit.TrainSet);
var largeLgbmMetric = context.Regression.Evaluate(largeLgbmEval, "y");

Console.WriteLine($"small lgbm rmse sur testset: {smallLgbmMetric.RootMeanSquaredError}, large lgbm rmse sur testset: {largeLgbmMetric.RootMeanSquaredError}");


[Cellule non executee - erreur environnement .NET]


## Utiliser AutoML pour simplifier la sélection des entraîneurs et l'optimisation des hyper-paramètres
La sélection des entraîneurs et l'optimisation des hyper-paramètres est un processus important avec beaucoup d'essais et d'erreurs. Ce processus peut être automatisé et simplifié en utilisant les capacités intégrées de `AutoMLExperiment`. `AutoMLExperiment` applique les dernières recherches de Microsoft Research pour effectuer une optimisation des hyper-paramètres rapide, précise et complète avec un budget de temps limité.

Le code ci-dessous montre comment utiliser `AutoMLExperiment` pour trouver le meilleur entraîneur ainsi que ses meilleurs hyper-paramètres sur le jeu de données non linéaire utilisé dans l'Exemple 2. Tout d'abord, un `SweepableEstimatorPipeline` est créé via `context.Auto().Regression("y")`, qui renvoie les régressions les plus populaires avec leur espace de recherche par défaut dans ML.Net. Ensuite, un `AutoMLExperiment` est créé. Il utilise `RootMeanSquaredError` comme métrique d'optimisation et la validation train-test pour évaluer le score des essais, et utilise `NotebookMonitor` pour présenter le processus d'entraînement. Une fois l'entraînement terminé, il renvoie le meilleur essai comme résultat.


In [5]:
using Microsoft.Data.Analysis;
using System;
using System.Linq;
using System.Collections.Generic;
using Microsoft.ML;
using Microsoft.ML.AutoML;
using Microsoft.ML.Data;
using Plotly.NET;
using Plotly.NET.CSharp;

// Définir le contexte ML
var context = new MLContext(seed: 1);

// Générer les données
var x = Enumerable.Range(-10000, 10000).Select(_x => _x * 0.1f).ToArray();
var y = x.Select(_x => 100 * _x * _x + (new Random(0).NextDouble() - 0.5) * 10).ToArray();

// Convertir 'y' en Single (float)
var y_single = y.Select(_y => (float)_y).ToArray();

// Créer un DataFrame
var df = new DataFrame();
df["X"] = DataFrameColumn.Create("X", x);
df["Label"] = DataFrameColumn.Create("Label", y_single); // Renommer en 'Label'

// Diviser le DataFrame en ensembles d'entraînement et de test
var trainTestSplit = context.Data.TrainTestSplit(df);

// Définir le pipeline de régression avec AutoML
var pipeline = context.Transforms.Concatenate("Features", "X")
    .Append(context.Auto().Regression("Label"));

// Classe NotebookMonitor pour suivre la progression AutoML (interface IMonitor mise à jour)
public class NotebookMonitor : IMonitor
{
    private readonly SweepablePipeline _pipeline;
    private readonly List<TrialResult> _completedTrials = new();
    private TrialResult? _bestTrial = null;
    private int _runningCount = 0;

    public NotebookMonitor(SweepablePipeline pipeline)
    {
        _pipeline = pipeline;
    }

    public void ReportRunningTrial(TrialSettings trialSettings)
    {
        _runningCount++;
        Console.WriteLine($"[En cours] Essai #{trialSettings.TrialId} en cours d'exécution...");
    }

    public void ReportCompletedTrial(TrialResult result)
    {
        _completedTrials.Add(result);
        var trialId = result.TrialSettings.TrialId;
        var metric = result.Metric;
        var duration = result.DurationInMilliseconds;
        Console.WriteLine($"[Complété] Essai #{trialId}: RMSE = {metric:F4} (durée: {duration}ms)");
    }

    public void ReportBestTrial(TrialResult result)
    {
        _bestTrial = result;
        Console.WriteLine($"\n=== Nouveau meilleur essai ===");
        Console.WriteLine($"Essai #{result.TrialSettings.TrialId}");
        Console.WriteLine($"RMSE: {result.Metric:F4}");
        Console.WriteLine($"Durée: {result.DurationInMilliseconds}ms");
    }

    public void ReportFailTrial(TrialSettings trialSettings, Exception exception)
    {
        Console.WriteLine($"[Échec] Essai #{trialSettings.TrialId}: {exception.Message}");
    }

    public void Display()
    {
        Console.WriteLine($"\n=== Résumé AutoML ===");
        Console.WriteLine($"Essais complétés: {_completedTrials.Count}");
        Console.WriteLine($"Essais en cours: {_runningCount - _completedTrials.Count}");
        
        if (_completedTrials.Any())
        {
            var best = _completedTrials.OrderBy(t => t.Metric).First();
            Console.WriteLine($"Meilleur RMSE: {best.Metric:F4}");
        }
        
        if (_bestTrial != null)
        {
            Console.WriteLine($"\nMeilleur essai final: #{_bestTrial.TrialSettings.TrialId}");
        }
    }
}

// Configurer le moniteur de Notebook
var monitor = new NotebookMonitor(pipeline);
var experiment = context.Auto().CreateExperiment();
experiment.SetPipeline(pipeline)
        .SetRegressionMetric(RegressionMetric.RootMeanSquaredError)
        .SetTrainingTimeInSeconds(30)
        .SetDataset(trainTestSplit.TrainSet, trainTestSplit.TestSet)
        .SetMonitor(monitor);

// Exécuter l'expérience
Console.WriteLine("Démarrage de l'expérience AutoML (30 secondes)...\n");
var res = await experiment.RunAsync();

// Obtenir le modèle
var model = res.Model;
var eval = model.Transform(trainTestSplit.TestSet);
var metric = context.Regression.Evaluate(eval, "Label");

// Afficher les résultats
monitor.Display();
Console.WriteLine($"\n=== Résultat final AutoML ===");
Console.WriteLine($"RMSE sur test set: {metric.RootMeanSquaredError:F4}");

[Cellule non executee - erreur environnement .NET]


## Exercices supplementaires

Les exercices suivants approfondissent les concepts d'entrainement, de selection d'algorithmes et d'hyperparametres.

### Exercice 1 : Comparaison systematique d'algorithmes

Entrainez le meme jeu de donnees avec 4 algorithmes differents, puis comparez leurs performances et temps d'entrainement.

**Objectifs** :
1. Utiliser les donnees lineaires (y = 100*x + bruit) definies plus haut dans le notebook
2. Entrainer 4 modeles : `SDCA`, `LightGbm`, `FastTree`, `AveragedPerceptron` (ou `OnlineGradientDescent`)
3. Mesurer le temps d'entrainement de chaque algorithme avec `Stopwatch`
4. Presenter un tableau comparatif : algorithme | RMSE train | RMSE test | temps (ms)
5. Interpreter les resultats : quel algorithme est le mieux adapte aux donnees lineaires et pourquoi

**Indice** : Les trainers sont accessibles via `context.Regression.Trainers.Sdca()`, `.LightGbm()`, `.FastTree()`, `.OnlineGradientDescent()`. Pour `AveragedPerceptron`, utilisez `context.BinaryClassification.Trainers.AveragedPerceptron()` si vous transformez le probleme en classification.

In [6]:
// Exercice 1 : Comparaison de 4 algorithmes
// TODO etudiant : Entrainez le meme dataset avec SDCA, LightGbm, FastTree et AveragedPerceptron
// Indice : chaque trainer est dans context.Regression.Trainers ou context.BinaryClassification.Trainers
// Etape 1 : utilisez les donnees lineaires (y = 100*x + bruit) definies plus haut
// Etape 2 : creez 4 pipelines avec le meme pretraitement mais des trainers differents
// Etape 3 : mesurez le temps d'entrainement avec Stopwatch pour chaque algorithme
// Etape 4 : comparez RMSE et temps dans un tableau resume

Console.WriteLine("Exercice a completer : comparaison de 4 algorithmes");

Exercice a completer


### Exercice 2 : Arret precoce et compromis performance/temps

Explorez le compromis entre nombre d'arbres, qualite du modele et temps d'entrainement. L'objectif est d'identifier le point optimal ou ajouter des arbres n'apporte plus de gain significatif.

**Objectifs** :
1. Creer un jeu de donnees de regression non-lineaire (quadratique avec bruit)
2. Entrainer `LightGbm` avec differentes valeurs de `numberOfTrees` (10, 50, 200)
3. Mesurer le temps d'entrainement avec `System.Diagnostics.Stopwatch`
4. Evaluer le RMSE sur train et test pour chaque configuration
5. Identifier le point de rendements decroissants

**Indice** : Utilisez `context.Regression.Trainers.LightGbm("y", numberOfTrees: 10)`. Comparez le gap entre RMSE train et RMSE test pour detecter le surapprentissage.

In [7]:
// Exercice 2 : Experience d'arret precoce (early stopping)
// TODO etudiant : Observez l'impact du nombre d'arbres sur les metriques et le temps d'entrainement
// Indice : FastTree et LightGbm acceptent numberOfLeaves, numberOfTrees comme hyperparametres
// Etape 1 : creez un jeu de donnees de regression non-lineaire (x, x*x + bruit)
// Etape 2 : entrainez LightGbm avec numberOfTrees=10, 50, 200 et mesurez le temps (Stopwatch)
// Etape 3 : evaluez le RMSE sur train et test pour chaque configuration
// Etape 4 : identifiez le point ou ajouter des arbres n'apporte plus de gain significatif

Console.WriteLine("Exercice a completer : arret precoce et nombre d'arbres");

Exercice a completer


Mise en pratique : comparaison de modeles.

In [8]:
// Exercice : Comparaison SDCA vs LightGbm sur données sinusoïdales

// TODO: Créer un MLContext avec seed fixe
MLContext sinusContext = null;  // TODO étudiant : remplacer par new MLContext(seed: 42)

// TODO: Générer les données sinusoïdales
// Indice: utilisez Enumerable.Range et MathF.Sin
var rand = new Random(42);
var xSinus = Enumerable.Range(0, 1000).Select(i => i * 0.01f).ToArray();
var ySinus = xSinus.Select(x => /* appliquez sin(x) + bruit */ 0f).ToArray();

// TODO: Créer le DataFrame
var dfSinus = new DataFrame();
// Ajoutez les colonnes X et y

// TODO: Diviser en train/test
var sinusSplit = sinusContext?.Data?.TrainTestSplit(dfSinus) ?? default;

// TODO: Créer le pipeline SDCA
IEstimator<ITransformer> sdcaSinusPipeline = null;

// TODO: Créer le pipeline LightGbm
IEstimator<ITransformer> lgbmSinusPipeline = null;

// TODO: Entraîner les deux modèles
ITransformer sdcaSinusModel = null;
ITransformer lgbmSinusModel = null;

// TODO: Évaluer SDCA sur train et test
RegressionMetrics sdcaSinusTrainMetric = null;
RegressionMetrics sdcaSinusTestMetric = null;

// TODO: Évaluer LightGbm sur train et test
RegressionMetrics lgbmSinusTrainMetric = null;
RegressionMetrics lgbmSinusTestMetric = null;

// Affichage des résultats
if (sinusContext != null && sdcaSinusTrainMetric != null && lgbmSinusTrainMetric != null)
{
    Console.WriteLine("=== Comparaison SDCA vs LightGbm sur données sinusoïdales ===");
    Console.WriteLine($"SDCA - RMSE Train: {sdcaSinusTrainMetric?.RootMeanSquaredError:F4}, RMSE Test: {sdcaSinusTestMetric?.RootMeanSquaredError:F4}");
    Console.WriteLine($"LightGbm - RMSE Train: {lgbmSinusTrainMetric?.RootMeanSquaredError:F4}, RMSE Test: {lgbmSinusTestMetric?.RootMeanSquaredError:F4}");
}
else
{
    Console.WriteLine("Exercice a completer : implementez les TODO pour comparer SDCA et LightGbm");
}

// TODO: Analysez les résultats - quel modèle est meilleur et pourquoi ?


Exercice a completer


## 🎯 Exercice : Comparaison d'algorithmes pour un nouveau jeu de données

Créez un jeu de données **sinusoïdal** (y = sin(x)) et comparez les performances de SDCA vs LightGbm sur ce type de données non-linéaires.

### Objectifs
1. Générer des données sinusoïdales avec du bruit aléatoire
2. Entraîner un modèle SDCA et un modèle LightGbm
3. Comparer les RMSE sur le jeu de test
4. Expliquer pourquoi un modèle performe mieux que l'autre

### Indices
- Utilisez `MathF.Sin(x)` pour générer les données
- SDCA est un algorithme linéaire, LightGbm est non-linéaire
- Comparez les RMSE sur train ET test pour détecter le surapprentissage

## Résumé

Ce notebook a couvert l'entraînement des modèles et AutoML avec ML.NET :

| Concept | Description |
|---------|-------------|
| SDCA | Entraîneur linéaire (Stochastic Dual Coordinate Ascent) |
| LightGBM | Entraîneur non-linéaire basé sur les arbres de décision |
| Surapprentissage | Modèle performant sur train mais pas sur test |
| Sous-apprentissage | Modèle insuffisamment puissant pour les données |
| AutoMLExperiment | Recherche automatique du meilleur algorithme et hyperparamètres |
| Hyperparamètre | Paramètre externe contrôlant le processus d'entraînement |

**Points clés** :
- Choisir le bon entraîneur dépend de la nature linéaire ou non-linéaire des données
- L'optimisation des hyperparamètres (HPO) est un processus itératif d'essais et d'erreurs
- AutoML simplifie la HPO en explorant automatiquement les configurations possibles
- Un bon modèle minimise la perte à la fois sur le train set et le test set

**Navigation** : [Index](README.md) | [<< ML-2 Data&Features](ML-2-Data%26Features.ipynb) | [Suivant >> ML-4 Evaluation](ML-4-Evaluation.ipynb)